# Chapter 14 - Recoveries: Salvage and Subrogation and Reinsurance
On this page, we will recreating Chapter 14 of Friendland.

We will begin by import packages and setting up some helper functions.

In [0]:
import numpy as np
import pandas as pd
import chainladder as cl
from IPython.display import display as nb_display


def format_exh(
    exh_df: pd.DataFrame,
    value_cols: list[str] = [],
    factor_cols: list[str] = [],
    other_formats: dict = {},
    origin_format: str | None = "{:%Y}",
    origin_name: str = "Accident Year",
    total_cols: list[str] = [],
):
    """Format exhibit"""

    def _format(
        df: pd.DataFrame,
        value_cols: list[str] = [],
        factor_cols: list[str] = [],
        other_formats: dict = {},
        origin_format: str = "{:%Y}",
        origin_name: str = "Accident Year",
    ):
        index_name = df.index.name if df.index.name else "index"
        return (
            df
            .reset_index()
            .rename(columns={index_name: origin_name})
            .style.hide(axis="index")
            .set_properties(**{"text-align": "right"})
            .format(
                {origin_name: origin_format}
                | {x: "{:,.0f}" for x in value_cols}
                | {x: "{:,.3f}" for x in factor_cols}
                | other_formats
            )
        )

    col_idx = pd.DataFrame(
        [[f"({i + 2})" for i in range(len(exh_df.columns))]],
        columns=exh_df.columns,
        index=["(1)"],
    )
    table_styler = _format(col_idx, origin_format=None, origin_name=origin_name).concat(
        _format(
            exh_df, value_cols, factor_cols, other_formats, origin_format, origin_name
        )
    )

    if len(total_cols) == 0:
        return table_styler
    else:
        return table_styler.concat(
            _format(
                exh_df.agg(["sum"]).rename(index={"sum": "Total"}),
                value_cols,
                factor_cols,
                {x: "" for x in exh_df.columns if x not in total_cols},
                None,
                origin_name,
            )
        )


def average_dev(tri: cl.Triangle, avg_params: dict[str, int]) -> dict[cl.Triangle]:
    """
    Create a dict of developed triangles for each of the selection assumptions on a given page
    """
    return {k: cl.Development(**v).fit_transform(tri) for k, v in avg_params.items()}


def combine_tri(devs: dict[cl.Triangle]) -> cl.Triangle:
    """Combine a dict of triangles into a singla triangle"""
    avgs = [v.rename("index", k) for k, v in devs.items()]
    return cl.concat(avgs, axis=0)


def combine_ldf(devs: dict[cl.Triangle]) -> cl.Triangle:
    """Combine the ldf_ of a dict of triangles into a singla triangle"""
    return combine_tri({k: v.ldf_ for k, v in devs.items()})


def dev_exh(
    tri: cl.Triangle,
    devs: dict[cl.Triangle],
    selected: cl.Triangle,
    pct_str: str | None = None,
) -> pd.DataFrame:
    """
    Print a Friedland development exhibit
    """
    print("PART 1 - Data Triangle")
    nb_display(tri)
    print("PART 2 - Age-to-Age Factors")
    nb_display(tri.age_to_age)
    print("PART 3 - Average Age-to-Age Factor")
    nb_display(combine_ldf(devs).to_frame().rename_axis(""))
    print("PART 4 - Selected Age-to-Age Factors")
    print("Selected")
    nb_display(selected.ldf_)
    print("CDF to Ultimate")
    nb_display(selected.cdf_)
    if pct_str:
        print(f"Percent {pct_str}")
        nb_display(1 / selected.cdf_)

## Exhibit I Analysis

In [0]:
# loading data and assumptions
e1_tri = cl.load_sample("friedland_auto_salsub") / 1000
e1_ss_assumptions = {}
e1_ss_assumptions["simple_5"] = {"n_periods": 5, "average": "simple"}
e1_ss_assumptions["simple_3"] = {"n_periods": 3, "average": "simple"}
e1_ss_assumptions["medial_5x1"] = {
    "n_periods": 5,
    "average": "simple",
    "drop_high": 1,
    "drop_low": 1,
}
e1_ss_assumptions["volume_5"] = {"n_periods": 5, "average": "volume"}
e1_ss_assumptions["volume_3"] = {"n_periods": 3, "average": "volume"}
e1_gross_assumptions = {}
e1_gross_assumptions["simple_5"] = {"n_periods": 5, "average": "simple"}
e1_gross_assumptions["simple_3"] = {"n_periods": 3, "average": "simple"}
e1_gross_assumptions["medial_5x1"] = {
    "n_periods": 5,
    "average": "simple",
    "drop_high": 1,
    "drop_low": 1,
}
e1_gross_assumptions["volume_5"] = {"n_periods": 5, "average": "volume"}
e1_gross_assumptions["volume_3"] = {"n_periods": 3, "average": "volume"}
e1_ratio_assumptions = {}
e1_ratio_assumptions["simple_5"] = {"n_periods": 5, "average": "simple"}
e1_ratio_assumptions["simple_3"] = {"n_periods": 3, "average": "simple"}
e1_ratio_assumptions["medial_5x1"] = {
    "n_periods": 5,
    "average": "simple",
    "drop_high": 1,
    "drop_low": 1,
}

# From Friedland p329
e1_repss_selection = "volume_5"
e1_recss_selection = "volume_5"
e1_grep_selection = "volume_5"
e1_gpaid_selection = "volume_5"
e1_ratio_selection = "medial_5x1"

# developing reported salv/sub
e1_repss_devs = average_dev(
    e1_tri["Reported Salvage and Subrogation"], e1_ss_assumptions
)
e1_repss_selected = cl.TailConstant(tail=1.0, projection_period=0).fit_transform(
    e1_repss_devs[e1_repss_selection]
)
e1_repss_cl = cl.Chainladder().fit(e1_repss_selected)

# developing received salv/sub
e1_recss_devs = average_dev(
    e1_tri["Received Salvage and Subrogation"], e1_ss_assumptions
)
e1_recss_selected = cl.TailConstant(tail=1.0, projection_period=0).fit_transform(
    e1_recss_devs[e1_recss_selection]
)
e1_recss_cl = cl.Chainladder().fit(e1_recss_selected)

# developing gross reported
e1_grep_devs = average_dev(e1_tri["Reported Claims"], e1_gross_assumptions)
e1_grep_selected = cl.TailConstant(tail=1.0, projection_period=0).fit_transform(
    e1_grep_devs[e1_grep_selection]
)
e1_grep_cl = cl.Chainladder().fit(e1_grep_selected)

# developing gross paid
e1_gpaid_devs = average_dev(e1_tri["Paid Claims"], e1_gross_assumptions)
e1_gpaid_selected = cl.TailConstant(tail=1.0, projection_period=0).fit_transform(
    e1_gpaid_devs[e1_gpaid_selection]
)
e1_gpaid_cl = cl.Chainladder().fit(e1_gpaid_selected)

# combining gross reported and paid
e1_gross_ult = (e1_grep_cl.ultimate_ + e1_gpaid_cl.ultimate_) / 2

# developing received ss to gross paid ratio
e1_ratio_tri = e1_tri["Received Salvage and Subrogation"] / e1_tri["Paid Claims"]

e1_ratio_devs = average_dev(e1_ratio_tri, e1_ratio_assumptions)
e1_ratio_selected = cl.TailConstant(tail=1.0, projection_period=0).fit_transform(
    e1_ratio_devs[e1_ratio_selection]
)
e1_ratio_cl = cl.Chainladder().fit(e1_ratio_selected)
e1_selected_ratio = e1_ratio_cl.ultimate_.copy()
e1_selected_ratio.loc[:, :, "2008", :] = 0.345
e1_ult_ss = e1_selected_ratio * e1_gross_ult

## Exhibit I Sheet 1

In [0]:
dev_exh(
    e1_tri["Reported Salvage and Subrogation"],
    e1_repss_devs,
    e1_repss_selected,
    "Reported",
)

## Exhibit I Sheet 2

In [0]:
dev_exh(
    e1_tri["Received Salvage and Subrogation"],
    e1_recss_devs,
    e1_recss_selected,
    "Received",
)

## Exhibit 1 Sheet 3

In [0]:
e1_repss_df = (
    cl
    .model_diagnostics(e1_repss_cl)
    .to_frame(keepdims=True, implicit_axis=True)
    .set_index("origin")
)
e1_recss_df = (
    cl
    .model_diagnostics(e1_recss_cl)
    .to_frame(keepdims=True, implicit_axis=True)
    .set_index("origin")
)
e1_s3 = pd.concat(
    [
        e1_repss_df["development"].rename("Age"),
        e1_repss_df["Latest"].rename("Reported S&S"),
        e1_recss_df["Latest"].rename("Received S&S"),
        e1_repss_df["CDF"].rename("Reported CDF"),
        e1_recss_df["CDF"].rename("Received CDF"),
        e1_repss_df["Ultimate"].rename("Ult S&S Using Rep"),
        e1_recss_df["Ultimate"].rename("Ult S&S Using Rec"),
    ],
    axis=1,
)
format_exh(
    e1_s3,
    [
        "Reported S&S",
        "Received S&S",
        "Ult S&S Using Rep",
        "Ult S&S Using Rec",
    ],
    [
        "Reported CDF",
        "Received CDF",
    ],
    total_cols=[
        "Reported S&S",
        "Received S&S",
        "Ult S&S Using Rep",
        "Ult S&S Using Rec",
    ],
)

## Exhibit I Sheet 4

In [0]:
dev_exh(
    e1_tri["Reported Claims"],
    e1_grep_devs,
    e1_grep_selected,
    "Reported",
)

## Exhibit I Sheet 5

In [0]:
dev_exh(
    e1_tri["Paid Claims"],
    e1_gpaid_devs,
    e1_gpaid_selected,
    "Paid",
)

## Exhibit I Sheet 6

In [0]:
e1_grep_df = (
    cl
    .model_diagnostics(e1_grep_cl)
    .to_frame(keepdims=True, implicit_axis=True)
    .set_index("origin")
)
e1_gpaid_df = (
    cl
    .model_diagnostics(e1_gpaid_cl)
    .to_frame(keepdims=True, implicit_axis=True)
    .set_index("origin")
)
e1_s6 = pd.concat(
    [
        e1_grep_df["development"].rename("Age"),
        e1_grep_df["Latest"].rename("Reported"),
        e1_gpaid_df["Latest"].rename("Paid"),
        e1_grep_df["CDF"].rename("Reported CDF"),
        e1_gpaid_df["CDF"].rename("Paid CDF"),
        e1_grep_df["Ultimate"].rename("Ult Gross Using Reported"),
        e1_gpaid_df["Ultimate"].rename("Ult Gross Using Paid"),
    ],
    axis=1,
)
e1_s6["Selected Ult Gross"] = e1_gross_ult.latest_diagonal.to_frame()
format_exh(
    e1_s6,
    [
        "Reported",
        "Paid",
        "Ult Gross Using Reported",
        "Ult Gross Using Paid",
        "Selected Ult Gross",
    ],
    [
        "Reported CDF",
        "Paid CDF",
    ],
    total_cols=[
        "Reported",
        "Paid",
        "Ult Gross Using Reported",
        "Ult Gross Using Paid",
        "Selected Ult Gross",
    ],
)

## Exhibit I Sheet 7

In [0]:
dev_exh(
    e1_ratio_tri,
    e1_ratio_devs,
    e1_ratio_selected,
)

## Exhibit I Sheet 8

In [0]:
e1_ratio_ult_df = (
    cl
    .model_diagnostics(e1_ratio_cl)
    .to_frame(keepdims=True, implicit_axis=True)
    .set_index("origin")
)
e1_s8 = e1_ratio_ult_df[["development", "Latest", "CDF", "Ultimate"]].rename(
    columns={
        "development": "Age",
        "Latest": "Ratio of Received S&S to Paid",
        "Ultimate": "Ult S&S Ratio",
    }
)
e1_s8["Selected S&S Ratio"] = e1_selected_ratio.latest_diagonal.to_frame()
e1_s8["Selected Ult Gross"] = e1_s6["Selected Ult Gross"]
e1_s8["Ult S&S"] = e1_ult_ss.latest_diagonal.to_frame()
format_exh(
    e1_s8,
    [
        "Selected Ult Gross",
        "Ult S&S",
    ],
    [
        "Ratio of Received S&S to Paid",
        "CDF",
        "Ult S&S Ratio",
        "Selected S&S Ratio",
    ],
    total_cols=[
        "Selected Ult Gross",
        "Ult S&S",
    ],
)

## Exhibit I Sheet 9

In [0]:
e1_s9 = pd.concat(
    [
        e1_recss_df["development"].rename("Age"),
        e1_recss_df["Latest"].rename("Received"),
        e1_repss_df["Ultimate"].rename("Ult S&S Using Reported"),
        e1_recss_df["Ultimate"].rename("Ult S&S Using Received"),
    ],
    axis=1,
)
e1_s9["Ult S&S Using Ratio"] = e1_ult_ss.latest_diagonal.to_frame()
e1_s9["Recoverable Using Reported"] = (
    e1_s9["Ult S&S Using Reported"] - e1_s9["Received"]
)
e1_s9["Recoverable Using Received"] = (
    e1_s9["Ult S&S Using Received"] - e1_s9["Received"]
)
e1_s9["Recoverable Using Ratio"] = e1_s9["Ult S&S Using Ratio"] - e1_s9["Received"]
format_exh(
    e1_s9,
    [
        "Received",
        "Ult S&S Using Reported",
        "Ult S&S Using Received",
        "Ult S&S Using Ratio",
        "Recoverable Using Reported",
        "Recoverable Using Received",
        "Recoverable Using Ratio",
    ],
    total_cols=[
        "Received",
        "Ult S&S Using Reported",
        "Ult S&S Using Received",
        "Ult S&S Using Ratio",
        "Recoverable Using Reported",
        "Recoverable Using Received",
        "Recoverable Using Ratio",
    ],
)

At the end of each exhibit, we reconcile to hardcoded figures from Friedland using a series of ``assert`` statemenets. When any of these statements errors out, we know some bug has been introduced into the package. 

In [0]:
# Exhibit I Sheet 1
assert np.allclose(
    e1_repss_selected.ldf_.values,
    np.array([
        1.068,
        0.998,
        1.000,
        1.000,
        1.000,
        1.001,
        1.000,
        1.000,
        1.000,
        1.000,
        1.000,
    ]),
    atol=0.001,
)
# Exhibit I Sheet 2
assert np.allclose(
    e1_recss_selected.ldf_.values,
    np.array([
        1.896,
        1.016,
        1.001,
        1.002,
        1.001,
        1.002,
        1.000,
        1.000,
        1.000,
        1.000,
        1.000,
    ]),
    atol=0.001,
)
# Exhibit I Sheet 3
assert np.allclose(
    e1_repss_cl.ultimate_.values.flatten(),
    np.array([793, 1360, 2421, 3637, 4091, 4370, 5165, 5737, 5720, 6025, 5776]),
    rtol=0.005,
)
assert np.allclose(
    e1_recss_cl.ultimate_.values.flatten(),
    np.array([793, 1360, 2421, 3637, 4090, 4374, 5175, 5760, 5688, 6088, 5252]),
    rtol=0.005,
)
# Exhibit I Sheet 4
assert np.allclose(
    e1_grep_selected.ldf_.values,
    np.array([
        1.114,
        1.001,
        1.000,
        1.000,
        1.000,
        1.000,
        1.000,
        1.000,
        1.000,
        1.000,
        1.000,
    ]),
    atol=0.001,
)
# Exhibit I Sheet 5
assert np.allclose(
    e1_gpaid_selected.ldf_.values,
    np.array([
        1.273,
        1.004,
        1.001,
        1.000,
        1.000,
        1.000,
        1.000,
        1.000,
        1.000,
        1.000,
        1.000,
    ]),
    atol=0.001,
)
# Exhibit I Sheet 6
assert np.allclose(
    e1_s6["Selected Ult Gross"].values,
    np.array([
        2864,
        4697,
        7902,
        10319,
        11137,
        12527,
        14536,
        16837,
        16952,
        16893,
        16453,
    ]),
    rtol=0.005,
)
# Exhibit I Sheet 7
assert np.allclose(
    e1_ratio_selected.ldf_.values,
    np.array([
        1.486,
        1.009,
        1.000,
        1.000,
        1.000,
        1.000,
        1.000,
        1.000,
        1.000,
        1.000,
        1.000,
    ]),
    atol=0.001,
)
# Exhibit I Sheet 8
assert np.allclose(
    e1_ratio_cl.ultimate_.values.flatten(),
    np.array([
        0.277,
        0.290,
        0.306,
        0.352,
        0.367,
        0.348,
        0.355,
        0.340,
        0.334,
        0.357,
        0.315,
    ]),
    atol=0.001,
)
assert np.allclose(
    e1_ult_ss.values.flatten(),
    np.array([793, 1360, 2421, 3637, 4090, 4365, 5165, 5731, 5658, 6036, 5676]),
    rtol=0.005,
)

## Exhibit 2 Sheet 1
This exhibit lays out common reinsurance structures but does not contain any IBNR estimation.

In [0]:
e2_s1_tri = cl.load_sample("friedland_qs")
nb_display(e2_s1_tri["Gross Reported Claims"])
nb_display(e2_s1_tri["Net Reported Claims"])
nb_display(e2_s1_tri["Net Reported Claims"] / e2_s1_tri["Gross Reported Claims"])

## Exhibit 2 Sheet 2

In [0]:
e2_s2_tri = cl.load_sample("friedland_xol")
nb_display(e2_s2_tri["Gross Reported Claims"])
nb_display(e2_s2_tri["Net Reported Claims"])
nb_display(e2_s2_tri["Gross Reported Claims"] - e2_s2_tri["Net Reported Claims"])

## Exhibit II Sheet 3
We will lay out the (primary) policy year and treaty (i.e. reinsurance policy) year tables separately

In [0]:
e2_s3_PY = pd.DataFrame(
    data=[
        ("2002 - 03", 1184999),
        ("2003 - 04", 1770725),
        ("2004 - 05", 1306107),
        ("2005 - 06", 2168077),
        ("2006 - 07", 1137216),
        ("2007 - 08", 1364048),
    ],
    columns=["Policy Year", "Net Gross Ult"],
).set_index("Policy Year")
format_exh(
    e2_s3_PY,
    ["Net Gross Ult"],
    origin_format=None,
    origin_name="Policy Year",
    total_cols=["Net Gross Ult"],
)

In [0]:
e2_s3_TY = pd.DataFrame(
    data=[
        ("2002 - 05", 4000000, 3753248, 3253624),
        ("2005 - 06", 1500000, 1500000, 1016783),
        ("2006 - 07", 1500000, 914262, 629296),
        ("2007 - 08", np.nan, 432679, 257877),
    ],
    columns=["Treaty Year", "Stop Loss Limit", "Net Net Reported", "Net Net Paid"],
).set_index("Treaty Year")
e2_s3_TY.insert(0, "Net Gross Ult", e2_s3_PY.iloc[3:, 0])
e2_s3_TY.loc["2002 - 05", "Net Gross Ult"] = e2_s3_PY.iloc[:3, 0].sum()
e2_s3_TY.insert(
    2, "Net Net Ult", e2_s3_TY[["Net Gross Ult", "Stop Loss Limit"]].min(axis=1)
)
e2_s3_TY["Net Net IBNR"] = e2_s3_TY["Net Net Ult"] - e2_s3_TY["Net Net Reported"]
e2_s3_TY["Net Net Unpaid"] = e2_s3_TY["Net Net Ult"] - e2_s3_TY["Net Net Paid"]
format_exh(
    e2_s3_TY,
    [
        "Net Gross Ult",
        "Net Net Ult",
        "Net Net Reported",
        "Net Net Paid",
        "Net Net IBNR",
        "Net Net Unpaid",
    ],
    origin_format=None,
    origin_name="Treaty Year",
    total_cols=[
        "Net Gross Ult",
        "Net Net Ult",
        "Net Net Reported",
        "Net Net Paid",
        "Net Net IBNR",
        "Net Net Unpaid",
    ],
)

In [0]:
# Exhibit II Sheet 3
assert np.all(
    e2_s3_TY["Net Net Ult"].values == np.array([4000000, 1500000, 1137216, 1364048])
)